<a href="https://colab.research.google.com/github/SelvaKumaran-G/flyrank-ml-internship3/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SelvaKumaran-G/flyrank-ml-internship3/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:

import pandas as pd
df = pd.read_csv('/content/content_refresh_anonymized.csv')
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## My rule

**In plain words:** A page is a refresh priority if its traffic is declining
AND its content is stale (hasn't been updated in a long time). Declining
traffic alone gets a lower "monitor" score, since it might resolve on its own.
Stale content alone with no decline isn't flagged, since there's no evidence
yet that it needs attention.

**Reason codes this rule can output:**
- `declining_and_stale` — traffic dropping and content is old
- `declining_only` — traffic dropping but content updated recently
- `stale_only` — content is old but traffic is stable/growing
- `no_signal` — neither condition met

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
import os
import numpy as np
import pandas as pd

def score_row(row):
    declining = row['trend_direction'] == 'down'
    # The column 'days_since_last_update' is missing from df. It needs to be added or handled.
    # For now, let's assume a default or placeholder if it's not critical for immediate execution
    # or if it will be added in a previous step.
    # For a temporary fix to allow execution, we'll assign a default value or skip this part if column not present.
    # However, for a proper fix, the 'days_since_last_update' column must be present in `df`.
    # Assuming for demonstration that this column will be present, or replacing with a placeholder for now.
    # If 'days_since_last_update' is not in row, this will cause a KeyError.
    # For this fix, I am leaving it as is, as the primary error was with page_id/client_hash_id.
    # A full solution would require defining or calculating 'days_since_last_update'.

    # TEMPORARY: Assuming 'days_since_last_update' will be available or handled elsewhere.
    # If it's not, the next line will raise a KeyError. You might need to mock or compute this column.
    stale = row.get('days_since_last_update', 0) > 180 # Using .get() for robustness, defaults to 0 if not found

    if declining and stale:
        score, reason, action = 3, "declining_and_stale", "refresh_priority"
    elif declining:
        score, reason, action = 2, "declining_only", "monitor"
    elif stale:
        score, reason, action = 1, "stale_only", "review_when_time_allows"
    else:
        score, reason, action = 0, "no_signal", "no_action"

    return pd.Series([score, reason, action])

df[['baseline_score', 'reason_code', 'action_label']] = df.apply(score_row, axis=1)

queue = df.sort_values('baseline_score', ascending=False)[
    ['content_id', 'client_id', 'baseline_score', 'reason_code', 'action_label']
]

os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"Wrote {len(queue)} rows to work/outputs/baseline_action_score.csv")
queue.head(10)

Wrote 30000 rows to work/outputs/baseline_action_score.csv


,content_id,client_id,baseline_score,reason_code,action_label
16417,content_f4b3081037b3,client_8722616204,3,declining_and_stale,refresh_priority
3280,content_a34d943a132c,client_d029fa3a95,3,declining_and_stale,refresh_priority
26799,content_77d4d5930e5e,client_7f2253d7e2,3,declining_and_stale,refresh_priority
26810,content_ecb6215e79fd,client_7f2253d7e2,3,declining_and_stale,refresh_priority
18652,content_0173fb0dc986,client_d029fa3a95,3,declining_and_stale,refresh_priority
22892,content_b8366b9ef4e8,client_6208ef0f77,3,declining_and_stale,refresh_priority
23741,content_df1fa766cac2,client_9400f1b21c,3,declining_and_stale,refresh_priority
26840,content_7f116ae1f6f5,client_9400f1b21c,3,declining_and_stale,refresh_priority
27374,content_3a93f78aa0a5,client_d029fa3a95,3,declining_and_stale,refresh_priority
16514,content_7368877ea310,client_7f2253d7e2,3,declining_and_stale,refresh_priority


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
top20 = queue.head(20)
top20

,content_id,client_id,baseline_score,reason_code,action_label
16417,content_f4b3081037b3,client_8722616204,3,declining_and_stale,refresh_priority
3280,content_a34d943a132c,client_d029fa3a95,3,declining_and_stale,refresh_priority
26799,content_77d4d5930e5e,client_7f2253d7e2,3,declining_and_stale,refresh_priority
26810,content_ecb6215e79fd,client_7f2253d7e2,3,declining_and_stale,refresh_priority
18652,content_0173fb0dc986,client_d029fa3a95,3,declining_and_stale,refresh_priority
22892,content_b8366b9ef4e8,client_6208ef0f77,3,declining_and_stale,refresh_priority
23741,content_df1fa766cac2,client_9400f1b21c,3,declining_and_stale,refresh_priority
26840,content_7f116ae1f6f5,client_9400f1b21c,3,declining_and_stale,refresh_priority
27374,content_3a93f78aa0a5,client_d029fa3a95,3,declining_and_stale,refresh_priority
16514,content_7368877ea310,client_7f2253d7e2,3,declining_and_stale,refresh_priority


## Top-20 review

1. page_id=[X] — refresh_priority, declining_and_stale. Flagged because
   traffic is dropping and content hasn't been touched in 180+ days.
   Would be wrong if the decline is seasonal rather than content quality.
2. page_id=[X] — refresh_priority, declining_and_stale. Same combined signal.
   Would be wrong if a competitor's new content — not staleness — caused the drop.
...
(continue for all 20 real rows from the printed table above — use actual
page_ids, don't leave placeholders)
20. page_id=[X] — ...

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak picks

Rows #[N] and #[N] both got `declining_and_stale` from thin margins — for
example a page just barely over the 180-day staleness cutoff. A page at
181 days scores identically to one at 500 days, which flattens real
differences in urgency. These are the picks I trust least in the top 20.

## Leakage check

- The rule only uses `trend_direction` and `days_since_last_update` — both
  observed at the same point-in-time snapshot, not derived from any future
  window.
- No product/CMS flags, no client names, and no post-hoc "did the refresh
  work" signal were used — this rule can't see outcomes, only current-state
  inputs.
- `trend_direction` is the same field the label (`is_declining_label`) is
  built from — using it directly in a hand-rule is fine here (this is a
  transparent baseline, not a model), but it means this rule can't be used
  as a *feature* in a later model without leaking the label.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.